# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BilaalBakare/Flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule, in plain words:

A page gets flagged if it's ranking poorly (position 21 or worse) and still gets a meaningful number of impressions (50 or more). This combination matters because my Signal 1 check showed CTR only truly collapses at position 21+ (0.13%, less than half of every better-ranked bucket) — so a page here is being seen but essentially never clicked. That's a real, fixable gap: the content exists and has visibility, it's just not converting that visibility into clicks.

The score is the page's impression count — the more people who see it and still don't click, the more potential traffic is being left on the table if the page were fixed, so higher impressions mean higher priority.

Reason code this rule can output: POOR_POSITION_WITH_VISIBILITY — the rule currently only checks for one condition, so it only ever produces this single reason code (as required — ONE reason code for this baseline).

Action label: improve_ctr — the recommended fix is improving the page's title/meta description or refreshing its content to better earn clicks at its current position, rather than a different action like removing the page or building new backlinks.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [2]:
import duckdb

con = duckdb.connect()

con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [3]:
q1 = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN '1-3'
            WHEN gsc_avg_position <= 10 THEN '4-10'
            WHEN gsc_avg_position <= 20 THEN '11-20'
            ELSE '21+'
        END AS position_bucket,
        COUNT(*) AS n,
        ROUND(SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0), 4) AS ctr
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03' AND gsc_avg_position IS NOT NULL
    GROUP BY position_bucket
    ORDER BY position_bucket
""").df()
q1

,position_bucket,n,ctr
0,1-3,727362,0.0038
1,11-20,519223,0.0031
2,21+,908354,0.0013
3,4-10,1456122,0.0032


In [4]:
q2 = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_impressions >= 1000 THEN 'high (1000+)'
            WHEN gsc_impressions >= 100 THEN 'medium (100-999)'
            ELSE 'low (<100)'
        END AS volume_bucket,
        COUNT(*) AS n,
        ROUND(AVG(gsc_avg_position), 2) AS avg_position
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
    GROUP BY volume_bucket
    ORDER BY volume_bucket
""").df()
q2

,volume_bucket,n,avg_position
0,high (1000+),32419,11.89
1,low (<100),9202770,16.85
2,medium (100-999),606189,11.02


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

rule_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_avg_position,
        gsc_impressions,
        gsc_clicks,
        gsc_impressions AS score,
        'POOR_POSITION_WITH_VISIBILITY' AS reason_code,
        'improve_ctr' AS action
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
      AND gsc_avg_position >= 21
      AND gsc_impressions >= 50
    ORDER BY score DESC
""").df()

rule_df.shape


(183158, 9)

In [6]:
import os
os.makedirs('work/outputs', exist_ok=True)
rule_df.to_csv('work/outputs/baseline_action_score.csv', index=False)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

rule_df.head(20)


,client_hash_id,content_hash_id,report_date,gsc_avg_position,gsc_impressions,gsc_clicks,score,reason_code,action
0,client_23a62021009f63c4,content_e6df0936699f5b8f,2026-03-31,25.035826,14682,269,14682,POOR_POSITION_WITH_VISIBILITY,improve_ctr
1,client_23a62021009f63c4,content_36e53e9c707674fc,2026-03-09,32.640344,9409,1,9409,POOR_POSITION_WITH_VISIBILITY,improve_ctr
2,client_23a62021009f63c4,content_3df3f32f3fd58dea,2026-03-10,22.399032,9300,14,9300,POOR_POSITION_WITH_VISIBILITY,improve_ctr
3,client_23a62021009f63c4,content_36e53e9c707674fc,2026-03-11,32.510609,9285,16,9285,POOR_POSITION_WITH_VISIBILITY,improve_ctr
4,client_23a62021009f63c4,content_3df3f32f3fd58dea,2026-03-12,24.985012,9274,10,9274,POOR_POSITION_WITH_VISIBILITY,improve_ctr
5,client_23a62021009f63c4,content_36e53e9c707674fc,2026-03-10,32.476180,8963,11,8963,POOR_POSITION_WITH_VISIBILITY,improve_ctr
6,client_23a62021009f63c4,content_36e53e9c707674fc,2026-03-08,33.982809,8842,3,8842,POOR_POSITION_WITH_VISIBILITY,improve_ctr
7,client_23a62021009f63c4,content_3df3f32f3fd58dea,2026-03-11,23.389476,8799,11,8799,POOR_POSITION_WITH_VISIBILITY,improve_ctr
8,client_23a62021009f63c4,content_3df3f32f3fd58dea,2026-03-09,24.421193,8267,10,8267,POOR_POSITION_WITH_VISIBILITY,improve_ctr
9,client_23a62021009f63c4,content_36e53e9c707674fc,2026-03-12,33.864952,8049,12,8049,POOR_POSITION_WITH_VISIBILITY,improve_ctr


Top-20 review

Note: because the grain is page-per-day, several pages repeat across multiple dates in this list (content_36e53e9c707674fc appears 8×, content_3df3f32f3fd58dea appears 6×) — this is really only 5 distinct pages, not 20 distinct opportunities. Worth flagging as a limitation of a daily-grain rule: without deduplicating to page-level (e.g., averaging or taking the latest date per page), the "top 20" overstates variety and could lead someone to review the same page 8 times.

content_e6df0936699f5b8f (pos 25.0, 14,682 impr, 269 clicks) — flagged for poor position despite visibility. Confidence: low — 269 clicks is a real CTR (~1.8%), much higher than the bucket average; this may already be performing reasonably and not need fixing. Wrong if: the page's actual CTR is fine relative to its query type/intent.
content_36e53e9c707674fc, 03-09 (pos 32.6, 9,409 impr, 1 click) — flagged for near-total visibility-to-click failure. Confidence: high — 1 click on 9,409 impressions is a stark, genuine gap. Wrong if: this was a one-day anomaly (e.g., a tracking glitch or a SERP feature that day suppressed clicks) rather than a persistent issue.
content_3df3f32f3fd58dea, 03-10 (pos 22.4, 9,300 impr, 14 clicks) — flagged for low CTR at a recoverable position. Confidence: medium — position 22 is close to page-2 boundary, a realistic CTR-fix candidate. Wrong if: the query is highly informational/non-commercial and low clicks are expected regardless of title quality.
content_36e53e9c707674fc, 03-11 (pos 32.5, 9,285 impr, 16 clicks) — same recurring page, slightly better day. Confidence: medium — shows this page's performance varies day to day. Wrong if: the rule should be looking at a monthly average for this page rather than flagging individual days separately.
content_3df3f32f3fd58dea, 03-12 (pos 25.0, 9,274 impr, 10 clicks) — same page as #3, different date. Confidence: medium — consistent underperformance across multiple days strengthens the case. Wrong if: duplicate flagging inflates perceived priority — this is really one page, not five separate opportunities.
content_36e53e9c707674fc, 03-10 (pos 32.5, 8,963 impr, 11 clicks) — repeat entry. Confidence: medium. Wrong if: same duplication concern as above.
content_36e53e9c707674fc, 03-08 (pos 34.0, 8,842 impr, 3 clicks) — repeat entry, worse CTR this day. Confidence: high for this specific day — 3 clicks on 8,842 impressions is a clear gap. Wrong if: this is noise within an otherwise more consistent page average.
content_3df3f32f3fd58dea, 03-11 (pos 23.4, 8,799 impr, 11 clicks) — repeat entry. Confidence: medium. Wrong if: same duplication concern.
content_3df3f32f3fd58dea, 03-09 (pos 24.4, 8,267 impr, 10 clicks) — repeat entry. Confidence: medium. Wrong if: same duplication concern.
content_36e53e9c707674fc, 03-12 (pos 33.9, 8,049 impr, 12 clicks) — repeat entry. Confidence: medium. Wrong if: same duplication concern.
content_36e53e9c707674fc, 03-05 (pos 31.2, 8,025 impr, 5 clicks) — repeat entry, weak CTR. Confidence: high for this day specifically. Wrong if: noise within the page's broader trend.
content_36e53e9c707674fc, 03-07 (pos 32.6, 7,973 impr, 15 clicks) — repeat entry, relatively better day. Confidence: low-medium — CTR here (~0.19%) is closer to bucket norms. Wrong if: this day alone suggests the page isn't as broken as other days imply.
content_36e53e9c707674fc, 03-04 (pos 33.2, 7,605 impr, 8 clicks) — repeat entry. Confidence: medium. Wrong if: same duplication concern.
content_9fff53e827550f9d (pos 30.5, 7,522 impr, 18 clicks) — a different client, first non-repeat page outside the dominant client. Confidence: low-medium — 18 clicks (~0.24% CTR) is close to bucket average, not a dramatic outlier. Wrong if: this is simply typical performance for its position, not a special opportunity.
content_36e53e9c707674fc, 03-06 (pos 33.2, 7,459 impr, 1 click) — repeat entry, near-zero CTR day. Confidence: high. Wrong if: one-day anomaly, as with entry #2.
content_8f06931116dbb8bf (pos 31.2, 7,435 impr, 40 clicks) — new page, notably higher clicks (~0.54% CTR). Confidence: low — this CTR is well above the bucket average; may not actually need fixing. Wrong if: the rule shouldn't have flagged this one at all — it's already outperforming peers at its position.
content_36e53e9c707674fc, 03-03 (pos 32.7, 7,174 impr, 9 clicks) — repeat entry. Confidence: medium. Wrong if: same duplication concern.
content_661a7734f691bef5 (pos 28.1, 7,026 impr, 0 clicks) — new page, zero clicks despite real visibility. Confidence: high — the clearest possible case of visibility without any conversion. Wrong if: this is a page that shouldn't be indexed/ranking at all (e.g., a thin or duplicate page), making "improve CTR" the wrong fix entirely — removal or noindex might be more appropriate.
content_3df3f32f3fd58dea, 03-04 (pos 27.2, 6,887 impr, 3 clicks) — repeat entry, weak day. Confidence: high for this day. Wrong if: noise within the page's broader trend.
content_3df3f32f3fd58dea, 03-05 (pos 24.5, 6,777 impr, 17 clicks) — repeat entry, better day. Confidence: low-medium. Wrong if: contradicts the pattern seen in nearby entries for the same page, suggesting high day-to-day noise rather than a stable problem.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.